In [ ]:
!pip install -q langchain
!pip install -q langchain-community
!pip install -q sentence-transformers
!pip install -q faiss-cpu
!pip install -q transformers
!pip install -q accelerate
!pip install -q bitsandbytes
!pip install -q pypdf

In [ ]:
!pip install -q langchain-text-splitters

In [ ]:
!pip install -U langchain
!pip install -U langchain-community

In [ ]:
hydroponics_text = """
Hydroponics is a method of growing plants without soil using nutrient-rich water.

The ideal pH range for lettuce in hydroponics is between 5.5 and 6.5.

NFT stands for Nutrient Film Technique. It uses a shallow stream of nutrient solution flowing over roots.

Deep Water Culture is a hydroponic system where plant roots are suspended in oxygenated nutrient water.

LED grow lights are commonly used in indoor hydroponic farming.

Hydroponics saves water compared to traditional farming methods.

Tomatoes grow well in hydroponic systems with pH between 5.5 and 6.5.

Hydroponic plants require nutrients such as nitrogen, phosphorus, potassium, calcium, and magnesium.

Poor oxygen supply can lead to root rot in hydroponic systems.

Coco peat and rockwool are commonly used growing mediums in hydroponics.

Drip systems slowly deliver nutrient solution directly to plant roots.

Hydroponics enables faster plant growth due to direct nutrient absorption.

Temperature control is important for healthy hydroponic crop growth.

Hydroponic farming is widely used in urban indoor agriculture.
"""

with open("hydroponics.txt", "w") as f:
    f.write(hydroponics_text)

print("Dataset Created Successfully")

In [ ]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("hydroponics.txt")

documents = loader.load()

print(documents)

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

texts = splitter.split_documents(documents)

print("Number of Chunks:", len(texts))

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embeddings Model Loaded")

In [ ]:
from langchain_community.vectorstores import FAISS

vector_db = FAISS.from_documents(
    texts,
    embeddings
)

print("FAISS Vector Database Created")

In [ ]:
from transformers import pipeline
from langchain_community.llms import HuggingFacePipeline

pipe = pipeline(
    "text-generation",
    model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    max_new_tokens=200,
    temperature=0.7
)

llm = HuggingFacePipeline(pipeline=pipe)

print("LLM Loaded Successfully")

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

retriever = vector_db.as_retriever(
    search_kwargs={"k": 3}
)

template = """
Answer the question based on the context below.

Context:
{context}

Question:
{question}
"""

prompt = ChatPromptTemplate.from_template(template)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG Chatbot Ready")

In [ ]:
query = "What is hydroponics?"

response = rag_chain.invoke(query)

print(response)

In [ ]:
print("HydroBot Chatbot Started")
print("Type 'exit' to stop\n")

while True:

    query = input("You: ")

    if query.lower() == "exit":
        print("Chatbot Stopped")
        break

    response = rag_chain.invoke(query)

    print("\nHydroBot:")
    print(response)
    print("\n")

HydroBot Chatbot Started
Type 'exit' to stop

